# AMEX Enterprise Credit Risk Platform
## Notebook 40 -- Dynamic / Behavioral Credit Scoring: Validation & Deployment
### Phase 3 . Problem Statement 6: Dynamic / Behavioral Credit Scoring

CRISP-DM stage: **Evaluation & Deployment**. Sprint 1, Notebook 3 of 4 for this problem. Depends on Problem 1 Notebooks 01-05 and this problem's own Notebooks 38-39 (reads `notebook_38_summary.json` for the policy and `notebook_39_summary.json` -> `dynamic_behavioral_scoring_modeling_results.json` for the real, measured per-W AUC-retention results).

**What this notebook does (real, computed on your machine when you run it):**
- Selects the trailing window to validate and deploy: the shortest candidate `W` that met Notebook 38's AUC-retention KPI, or -- if none did -- the best-performing candidate, carried through and clearly flagged **NOT RECOMMENDED FOR PRODUCTION** rather than hidden
- Rebuilds that `W`'s trailing-window feature set and retrains the final model, deterministically reproducing Notebook 39's reported metrics (checked, not assumed) -- **and reproduces the FULL classification metrics suite** (ROC-AUC, PR-AUC, Log Loss, AMEX metric, plus Accuracy/Precision/Recall/F1/Specificity/MCC/confusion matrix at both the 0.5 and F1-optimal thresholds), per the standing metrics-suite directive
- Runs real statistical validation: 2,000-resample bootstrap confidence intervals on both holdout AUC and PR-AUC, a real calibration check (predicted-PD deciles vs. observed default rate), and a split-half population-stability (PSI) check on the predicted score
- Assembles a full metrics-suite statistical validation summary table (21 rows -- every threshold-free and threshold-dependent metric, both measured and cross-checked)
- States an explicit, honest deployment-scope limitation (this is a RECENCY score using only a customer's most recent `W` statements -- complementary to, not a replacement for, Problem 1's full-history champion; plus the data-limitation and feature-leakage notes carried from Notebook 38)
- Persists the model + preprocessing artifacts, generates a real, runnable FastAPI scoring service (`dynamic_behavioral_service.py`), and proves it live via a self-test that imports the exact file just written to disk and checks its output against a direct computation
- Produces a Word report (`Dynamic_Behavioral_Validation_Deployment_Report.docx`) combining the full metrics-suite validation summary, honest limitations, deployment readiness checklist, and charts -- including Notebook 39's ROC and Precision-Recall curves, reused directly rather than regenerated, so the same curves shown in that notebook also appear in this report

**What this notebook does NOT do:** financial-impact reporting and final repository packaging -- that's Notebook 41.

Zero-fabrication: every metric in this notebook is computed live from your real Kaggle data and Notebook 39's real results on this run -- including the possibility, reported plainly if it happens, that no candidate window met the KPI target.

In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD CONFIG FROM NOTEBOOKS 01-05, 38, 39
# =============================================================================
import os
import sys
import gc
import json
import time
import warnings
import importlib.util
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Config From Notebooks 01-05, 38, 39")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB02_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_02_summary.json"
NB05_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_05_summary.json"
NB38_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_38_summary.json"
NB39_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_39_summary.json"

for _p, _fix in [
    (CONFIG_PATH, "run 01_business_understanding.ipynb first"),
    (NB02_SUMMARY_PATH, "run 02_data_engineering.ipynb first"),
    (NB05_SUMMARY_PATH, "run 05_model_development.ipynb first"),
    (NB38_SUMMARY_PATH, "run 38_dynamic_behavioral_scoring_business_understanding.ipynb first"),
    (NB39_SUMMARY_PATH, "run 39_dynamic_behavioral_scoring_modeling.ipynb first -- this notebook "
                         "consumes its real per-W AUC-retention results, not a guess"),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found.\nFix: {_fix}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)
with open(NB02_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB02_SUMMARY = json.load(f)
with open(NB05_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB05_SUMMARY = json.load(f)
with open(NB38_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB38_SUMMARY = json.load(f)
with open(NB39_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB39_SUMMARY = json.load(f)

POLICY_PATH = Path(NB38_SUMMARY["policy_path"])
if not POLICY_PATH.exists():
    raise FileNotFoundError(f"{POLICY_PATH} not found.\nFix: re-run Notebook 38.")
with open(POLICY_PATH, "r", encoding="utf-8") as f:
    DBS_POLICY = json.load(f)
DBS_FEATURE_LIST = sorted(DBS_POLICY["feature_space"]["features"])

MODELING_RESULTS_PATH = Path(NB39_SUMMARY["modeling_results_path"])
if not MODELING_RESULTS_PATH.exists():
    raise FileNotFoundError(f"{MODELING_RESULTS_PATH} not found.\nFix: re-run 39_dynamic_behavioral_scoring_modeling.ipynb.")
with open(MODELING_RESULTS_PATH, "r", encoding="utf-8") as f:
    MODELING_ARTIFACT = json.load(f)

RESULTS_BY_W = {int(w): v for w, v in MODELING_ARTIFACT["results_by_w"].items()}
WS_MEETING_KPI = MODELING_ARTIFACT["ws_meeting_kpi_target"]
KPI_TARGETS = MODELING_ARTIFACT["kpi_targets"]
FULL_HISTORY_AUC = MODELING_ARTIFACT["full_history_reference_auc"]
FULL_HISTORY_AMEX_METRIC = MODELING_ARTIFACT.get("full_history_reference_amex_metric")
NB39_ROC_CHART_PATH = Path(MODELING_ARTIFACT["roc_chart_path"])
NB39_PR_CHART_PATH = Path(MODELING_ARTIFACT["pr_chart_path"])

# --- Pick the trailing window this notebook validates and deploys. If one or
#     more candidates met Notebook 38's KPI target, the SHORTEST (fewest
#     trailing statements required) one wins -- a shorter trailing window
#     covers more of the customer book (Notebook 38's real coverage numbers
#     are monotonically decreasing in W) and reflects MORE RECENT behavior
#     with less lag, both operationally preferable, all else equal. If NONE
#     met the KPI, this is reported plainly (not hidden): the best-AUC
#     candidate is still carried through validation and deployment-packaging
#     for completeness, but flagged NOT RECOMMENDED FOR PRODUCTION throughout
#     -- the same honest "not yet viable" standard Notebook 36 held Problem
#     5's own equivalent selection to. ---
if WS_MEETING_KPI:
    WINNING_W = min(WS_MEETING_KPI)
    MEETS_KPI = True
    print(f"Candidates meeting the {KPI_TARGETS['min_auc_retention_vs_full_history']:.0%} AUC-retention KPI: {WS_MEETING_KPI}")
    print(f"Selected WINNING_W = {WINNING_W} (shortest trailing window that still clears the KPI)")
else:
    WINNING_W = max(RESULTS_BY_W.keys(), key=lambda w: RESULTS_BY_W[w]["holdout_auc"])
    MEETS_KPI = False
    print(
        f"HONEST FINDING (carried forward from Notebook 39): none of the candidate trailing windows "
        f"{sorted(RESULTS_BY_W.keys())} met the {KPI_TARGETS['min_auc_retention_vs_full_history']:.0%} "
        f"AUC-retention KPI on the real run. Selected WINNING_W = {WINNING_W} (best holdout AUC among "
        f"candidates, {RESULTS_BY_W[WINNING_W]['holdout_auc']:.4f}) so this notebook can still validate and "
        f"package it -- but every section below marks it NOT RECOMMENDED FOR PRODUCTION. This is a real "
        f"outcome of this run's real data, not a defect in this notebook."
    )

WINNING_W_RESULT = RESULTS_BY_W[WINNING_W]
print(f"\nWINNING_W = {WINNING_W}")
print(json.dumps(WINNING_W_RESULT, indent=2))

PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
DETECTED_LOGICAL_CORES = PROJECT_CONFIG["hardware"]["logical_cores_detected"]
RANDOM_SEED = PROJECT_CONFIG["random_seed"]
_resource_limits = PROJECT_CONFIG.get("resource_limits", {})
WARP_THREAD_COUNT = (
    _resource_limits.get("warp_thread_count")
    or PROJECT_CONFIG.get("warp_thread_count")
    or DETECTED_LOGICAL_CORES
)
MAX_RAM_BYTES = _resource_limits.get("max_ram_bytes")

CHAMPION_NAME = NB05_SUMMARY["champion_model"]
CHAMPION_METRICS = NB05_SUMMARY["champion_metrics"]

if "dynamic_behavioral_scoring_deployment" in PILLAR_DIRS:
    DBS_DEPLOYMENT_DIR = PILLAR_DIRS["dynamic_behavioral_scoring_deployment"]
else:
    DBS_DEPLOYMENT_DIR = (
        PROJECT_ROOT / "Phase3_Behavioral_Intelligence"
        / "Problem6_Dynamic_Behavioral_Credit_Scoring" / "deployment"
    )
    print(f"NOTE: 'dynamic_behavioral_scoring_deployment' not in pillar_dirs -- using fallback: {DBS_DEPLOYMENT_DIR}")
DBS_DEPLOYMENT_DIR.mkdir(parents=True, exist_ok=True)
API_SUBDIR = DBS_DEPLOYMENT_DIR / "api"
API_SUBDIR.mkdir(parents=True, exist_ok=True)
MODELS_SUBDIR = DBS_DEPLOYMENT_DIR / "models"
MODELS_SUBDIR.mkdir(parents=True, exist_ok=True)

print(f"\nDeployment artifacts will be written under: {DBS_DEPLOYMENT_DIR}")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION & LIBRARY IMPORTS
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration & Library Imports")

os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import polars as pl
except ImportError:
    missing.append("polars")
try:
    import numpy as np
except ImportError:
    missing.append("numpy")
try:
    import pandas as pd
except ImportError:
    missing.append("pandas")
try:
    import psutil
except ImportError:
    missing.append("psutil")
try:
    import joblib
except ImportError:
    missing.append("joblib")
try:
    from sklearn.metrics import (
        roc_auc_score, average_precision_score, accuracy_score, precision_score,
        recall_score, f1_score, log_loss, matthews_corrcoef, confusion_matrix,
        roc_curve, precision_recall_curve,
    )
except ImportError:
    missing.append("scikit-learn")
try:
    from xgboost import XGBClassifier
except ImportError:
    missing.append("xgboost")
try:
    from fastapi.testclient import TestClient
except ImportError:
    missing.append("fastapi")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    missing.append("matplotlib")
try:
    from docx import Document
    from docx.shared import Inches
    from docx.enum.text import WD_ALIGN_PARAGRAPH
except ImportError:
    missing.append("python-docx")
try:
    import importlib.metadata as importlib_metadata
except ImportError:
    import importlib_metadata

if missing:
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\n"
        "Fix: run this in a terminal, then re-run this cell:\n"
        f"    pip install {' '.join(missing)}"
    )


def _rss_gb() -> float:
    return psutil.Process().memory_info().rss / 1e9


def amex_metric_numpy(y_true: "np.ndarray", y_pred: "np.ndarray") -> float:
    """Official AMEX competition metric -- same implementation as Notebook 05
    Section 3 / Notebook 39 Section 2, reused here for consistency."""
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    def top_four_percent_captured(yt, yp):
        order = np.argsort(-yp, kind="mergesort")
        yt_sorted = yt[order]
        weight = np.where(yt_sorted == 0, 20.0, 1.0)
        cum_weight = np.cumsum(weight)
        cutoff = 0.04 * weight.sum()
        mask = cum_weight <= cutoff
        total_pos = yt_sorted.sum()
        if total_pos == 0:
            return 0.0
        return float(yt_sorted[mask].sum() / total_pos)

    def weighted_gini(yt, yp):
        order = np.argsort(-yp, kind="mergesort")
        yt_sorted = yt[order]
        weight = np.where(yt_sorted == 0, 20.0, 1.0)
        random_cum = np.cumsum(weight / weight.sum())
        total_pos_weighted = (yt_sorted * weight).sum()
        if total_pos_weighted == 0:
            return 0.0
        cum_pos_found = np.cumsum(yt_sorted * weight)
        lorentz = cum_pos_found / total_pos_weighted
        return float(((lorentz - random_cum) * weight).sum())

    g_actual = weighted_gini(y_true, y_pred)
    g_perfect = weighted_gini(y_true, y_true)
    normalized_gini = g_actual / g_perfect if g_perfect != 0 else 0.0
    top4 = top_four_percent_captured(y_true, y_pred)
    return 0.5 * (normalized_gini + top4)


logger.info(f"Polars thread pool configured to {os.environ['POLARS_MAX_THREADS']} threads (95% cap, WARP 6.4)")
print(f"Process RSS at Section 2 start: {_rss_gb():.2f} GB")
if MAX_RAM_BYTES:
    print(f"Configured RAM ceiling (90% of detected total): {MAX_RAM_BYTES / 1e9:.1f} GB")
print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: RESOLVE REAL DATA PATHS
# =============================================================================
_section("SECTION 3: Resolve Real Data Paths")


def _resolve_pillar_file(filename: str, pillar_key: str, legacy_folder_name: str,
                          stored_path_str: str = None, min_size: int = 10_000) -> Path:
    _candidates = [
        PROJECT_ROOT / "Phase1_Foundation" / "Problem1_Credit_Scoring_PD_Prediction"
        / legacy_folder_name / filename,
    ]
    if pillar_key in PILLAR_DIRS:
        _candidates.append(PILLAR_DIRS[pillar_key] / filename)
    _candidates.append(PROJECT_ROOT / legacy_folder_name / filename)
    if stored_path_str:
        _candidates.append(Path(stored_path_str))
    for _c in _candidates:
        if _c.exists() and _c.stat().st_size > min_size:
            return _c
    raise FileNotFoundError(
        f"Could not resolve a real, non-trivial {filename}. Checked:\n"
        + "\n".join(f"  - {c}" for c in _candidates)
        + "\nFix: run the notebook that produces this file again, or tell me the real path."
    )


_raw_candidates = []
if "raw_data_dir" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["raw_data_dir"]) / "train_data.csv")
if "data_root" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["data_root"]) / "train_data.csv")
_raw_candidates.append(PROJECT_ROOT.parent / "Raw Data From Kaggle" / "train_data.csv")

RAW_TRAIN_DATA_PATH = None
for _candidate in _raw_candidates:
    if _candidate.exists() and _candidate.stat().st_size > 1_000_000:
        RAW_TRAIN_DATA_PATH = _candidate
        break
if RAW_TRAIN_DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find the raw train_data.csv. Checked:\n" + "\n".join(f"  - {c}" for c in _raw_candidates)
    )
RAW_TRAIN_LABELS_PATH = RAW_TRAIN_DATA_PATH.parent / "train_labels.csv"
if not RAW_TRAIN_LABELS_PATH.exists():
    raise FileNotFoundError(f"{RAW_TRAIN_LABELS_PATH} not found.")

TRAIN_SPLIT_PATH = _resolve_pillar_file(
    "train_split.csv", "data_engineering", "Data_Engineering",
    stored_path_str=NB02_SUMMARY.get("output_files", {}).get("train_split.csv"),
)
TEST_SPLIT_PATH = _resolve_pillar_file(
    "test_split.csv", "data_engineering", "Data_Engineering",
    stored_path_str=NB02_SUMMARY.get("output_files", {}).get("test_split.csv"),
)

print(f"Raw train_data.csv  : {RAW_TRAIN_DATA_PATH}")
print(f"train_split.csv     : {TRAIN_SPLIT_PATH}")
print(f"test_split.csv      : {TEST_SPLIT_PATH}")
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: LIVE SCHEMA DETECTION -- BASE RAW COLUMNS (SAME AS NOTEBOOK 39)
# =============================================================================
_section("SECTION 4: Live Schema Detection -- Base Raw Columns")

with open(RAW_TRAIN_DATA_PATH, "r", encoding="utf-8") as f:
    _train_header = f.readline().strip().split(",")
_header_cols = set(_train_header)

_SUFFIXES = ("_trend_slope", "_trend_delta", "_last")
BASE_FEATURE_COLUMNS = set()
for _feat in DBS_FEATURE_LIST:
    for _suf in _SUFFIXES:
        if _feat.endswith(_suf):
            BASE_FEATURE_COLUMNS.add(_feat[: -len(_suf)])
            break

_missing_base_cols = BASE_FEATURE_COLUMNS - _header_cols
if _missing_base_cols:
    raise RuntimeError(
        f"{len(_missing_base_cols)} base column(s) from the reused feature list are "
        f"not present in the real raw CSV header: {sorted(_missing_base_cols)}"
    )
BASE_FEATURE_COLUMNS = sorted(BASE_FEATURE_COLUMNS)
print(f"Real base D_* raw columns needed: {len(BASE_FEATURE_COLUMNS)}")
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: TRAILING-WINDOW FEATURE ENGINEERING FUNCTION (SAME AS NOTEBOOK 39)
# =============================================================================
_section("SECTION 5: Trailing-Window Feature Engineering Function")


def build_trailing_window_store(csv_path: Path, base_cols: list, w: int) -> "pl.DataFrame":
    """Identical to Notebook 39's build_trailing_window_store() -- re-declared
    here (this platform's notebooks are not yet wired to a shared module, so
    reproducible-by-construction logic like this is duplicated verbatim
    rather than imported; see root ROADMAP.md)."""
    schema_overrides = {"customer_ID": pl.Utf8, "S_2": pl.Utf8}
    for c in base_cols:
        schema_overrides[c] = pl.Float32
    _inf_clean_exprs = [
        pl.when(pl.col(c).is_infinite()).then(None).otherwise(pl.col(c)).alias(c)
        for c in base_cols
    ]
    lf = (
        pl.scan_csv(str(csv_path), schema_overrides=schema_overrides)
        .with_columns(pl.col("S_2").str.to_date("%Y-%m-%d"))
        .with_columns(_inf_clean_exprs)
        .sort(["customer_ID", "S_2"])
        .with_columns(
            pl.len().over("customer_ID").alias("_n_statements"),
            pl.int_range(pl.len()).over("customer_ID").alias("_row_idx"),
        )
        .filter(pl.col("_row_idx") >= (pl.col("_n_statements") - w))
        .with_columns(pl.int_range(pl.len()).over("customer_ID").cast(pl.Float32).alias("_t_idx"))
    )
    agg_exprs = [pl.len().alias("_actual_window_len")]
    for c in base_cols:
        agg_exprs += [
            pl.cov(pl.col("_t_idx"), pl.col(c)).alias(f"_cov_{c}"),
            pl.when(pl.col(c).is_not_null()).then(pl.col("_t_idx")).otherwise(None)
              .var().alias(f"_var_t_{c}"),
            pl.col(c).first().alias(f"_first_{c}"),
            pl.col(c).last().alias(f"{c}_last"),
        ]
    grouped = lf.group_by("customer_ID", maintain_order=False).agg(agg_exprs)
    _trend_exprs = []
    for c in base_cols:
        _trend_exprs.append(
            pl.when((pl.col(f"_var_t_{c}").is_not_null()) & (pl.col(f"_var_t_{c}") > 0))
            .then(pl.col(f"_cov_{c}") / pl.col(f"_var_t_{c}")).otherwise(None).alias(f"{c}_trend_slope")
        )
        _trend_exprs.append((pl.col(f"{c}_last") - pl.col(f"_first_{c}")).alias(f"{c}_trend_delta"))
    _keep_cols = ["customer_ID", "_actual_window_len"]
    _keep_cols += [f"{c}_last" for c in base_cols]
    _keep_cols += [f"{c}_trend_slope" for c in base_cols] + [f"{c}_trend_delta" for c in base_cols]
    result = grouped.with_columns(_trend_exprs).select(_keep_cols).sort("customer_ID")
    return result.collect(engine="streaming")


print("build_trailing_window_store() re-declared (identical to Notebook 39).")
print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: REBUILD WINNING W'S FEATURE SET & RETRAIN THE FINAL MODEL --
#            FULL CLASSIFICATION METRICS SUITE REPRODUCED
# =============================================================================
_section(f"SECTION 6: Rebuild W={WINNING_W}'s Feature Set & Retrain the Final Model")

# --- Notebook 39 evaluated every W but did not persist a fitted model object
#     for each one (only the holdout metrics + ROC/PR curve points). This
#     retrains ONLY the winning W, deterministically (same RANDOM_SEED, same
#     data, same hyperparameters Notebook 39 used) -- not a new decision, a
#     real reproduction of the exact model whose metrics are already
#     reported above, now with the full classification metrics suite
#     recomputed and cross-checked against Notebook 39's reported numbers. ---
labels_df = pl.read_csv(str(RAW_TRAIN_LABELS_PATH), schema_overrides={"customer_ID": pl.Utf8, "target": pl.Int8})
train_ids_set = set(pl.read_csv(str(TRAIN_SPLIT_PATH), columns=["customer_ID"])["customer_ID"].to_list())
val_ids_set = set(pl.read_csv(str(TEST_SPLIT_PATH), columns=["customer_ID"])["customer_ID"].to_list())

_t0 = time.time()
_base = build_trailing_window_store(RAW_TRAIN_DATA_PATH, BASE_FEATURE_COLUMNS, WINNING_W)
_base = _base.filter(pl.col("_actual_window_len") == WINNING_W)
engineered = _base.join(labels_df, on="customer_ID", how="inner")
print(f"Rebuilt {engineered.shape[0]:,} customers x {engineered.shape[1]} columns in {time.time() - _t0:.1f}s")
del _base
gc.collect()

train_df = engineered.filter(pl.col("customer_ID").is_in(train_ids_set))
holdout_df = engineered.filter(pl.col("customer_ID").is_in(val_ids_set))
holdout_customer_ids = holdout_df.get_column("customer_ID").to_list()
del engineered
gc.collect()

all_feature_cols = [c for c in DBS_FEATURE_LIST if c in train_df.columns]
if len(all_feature_cols) != len(DBS_FEATURE_LIST):
    raise RuntimeError(f"Only {len(all_feature_cols)}/{len(DBS_FEATURE_LIST)} reused features were rebuilt.")

_inf_clean_exprs = [
    pl.when(pl.col(c).is_infinite() | pl.col(c).is_nan()).then(None).otherwise(pl.col(c)).cast(pl.Float32).alias(c)
    for c in all_feature_cols
]
train_df = train_df.with_columns(_inf_clean_exprs)
holdout_df = holdout_df.with_columns(_inf_clean_exprs)

feature_medians = train_df.select(
    [pl.col(c).median().fill_null(0.0).alias(c) for c in all_feature_cols]
).to_dicts()[0]
_impute_exprs = [pl.col(c).fill_null(feature_medians[c]) for c in all_feature_cols]
train_df = train_df.with_columns(_impute_exprs)
holdout_df = holdout_df.with_columns(_impute_exprs)

X_train = train_df.select(all_feature_cols).to_numpy().astype(np.float32, copy=False)
y_train = train_df.get_column("target").to_numpy().astype(np.int64, copy=False)
X_holdout = holdout_df.select(all_feature_cols).to_numpy().astype(np.float32, copy=False)
y_holdout = holdout_df.get_column("target").to_numpy().astype(np.int64, copy=False)
del train_df, holdout_df
gc.collect()

final_model = XGBClassifier(
    n_estimators=400, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
    tree_method="hist", n_jobs=WARP_THREAD_COUNT, random_state=RANDOM_SEED,
    eval_metric="auc", verbosity=0,
)
final_model.fit(X_train, y_train)
proba_train = final_model.predict_proba(X_train)[:, 1]
proba_holdout = final_model.predict_proba(X_holdout)[:, 1]

# --- Threshold-free metrics ---
reproduced_auc = float(roc_auc_score(y_holdout, proba_holdout))
reproduced_pr_auc = float(average_precision_score(y_holdout, proba_holdout))
reproduced_log_loss = float(log_loss(y_holdout, proba_holdout, labels=[0, 1]))
reproduced_amex = float(amex_metric_numpy(y_holdout, proba_holdout))
holdout_positive_rate = float(y_holdout.mean())

fpr, tpr, _roc_thresholds = roc_curve(y_holdout, proba_holdout)
pr_precision, pr_recall, _pr_thresholds = precision_recall_curve(y_holdout, proba_holdout)

_f1_scores = np.where(
    (pr_precision + pr_recall) > 0,
    2 * pr_precision * pr_recall / np.where((pr_precision + pr_recall) > 0, pr_precision + pr_recall, 1.0),
    0.0,
)
_best_idx = int(np.argmax(_f1_scores[:-1])) if len(_f1_scores) > 1 else 0
f1_optimal_threshold = float(_pr_thresholds[_best_idx]) if len(_pr_thresholds) > 0 else 0.5


def _threshold_metrics(threshold: float) -> dict:
    pred = (proba_holdout >= threshold).astype(np.int64)
    tn, fp, fn, tp = confusion_matrix(y_holdout, pred, labels=[0, 1]).ravel()
    specificity = float(tn / (tn + fp)) if (tn + fp) > 0 else 0.0
    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_holdout, pred)),
        "precision": float(precision_score(y_holdout, pred, zero_division=0)),
        "recall": float(recall_score(y_holdout, pred, zero_division=0)),
        "f1": float(f1_score(y_holdout, pred, zero_division=0)),
        "specificity": specificity,
        "mcc": float(matthews_corrcoef(y_holdout, pred)),
        "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
    }


metrics_at_050 = _threshold_metrics(0.5)
metrics_at_f1_optimal = _threshold_metrics(f1_optimal_threshold)

print(f"Reproduced holdout AUC     : {reproduced_auc:.4f}  (Notebook 39 reported {WINNING_W_RESULT['holdout_auc']:.4f})")
print(f"Reproduced holdout PR-AUC  : {reproduced_pr_auc:.4f}  (Notebook 39 reported {WINNING_W_RESULT['holdout_pr_auc']:.4f})")
print(f"Reproduced holdout LogLoss : {reproduced_log_loss:.4f}  (Notebook 39 reported {WINNING_W_RESULT['holdout_log_loss']:.4f})")
print(f"Reproduced holdout AMEX    : {reproduced_amex:.4f}  (Notebook 39 reported {WINNING_W_RESULT['holdout_amex_metric']:.4f})")
_reproduction_matches = (
    abs(reproduced_auc - WINNING_W_RESULT["holdout_auc"]) < 1e-6
    and abs(reproduced_pr_auc - WINNING_W_RESULT["holdout_pr_auc"]) < 1e-6
)
print(f"Reproduction matches Notebook 39 exactly (same seed, same data): {_reproduction_matches}")
print(f"\n{'threshold':>10} {'accuracy':>9} {'precision':>10} {'recall':>8} {'f1':>7} "
      f"{'specificity':>11} {'mcc':>7}  confusion(tn,fp,fn,tp)")
for _label, _m in (("0.50 (standard)", metrics_at_050), (f"{f1_optimal_threshold:.3f} (F1-optimal)", metrics_at_f1_optimal)):
    _cm = _m["confusion_matrix"]
    print(f"{_label:>10} {_m['accuracy']:>9.4f} {_m['precision']:>10.4f} {_m['recall']:>8.4f} "
          f"{_m['f1']:>7.4f} {_m['specificity']:>11.4f} {_m['mcc']:>7.4f}  "
          f"({_cm['tn']:,}, {_cm['fp']:,}, {_cm['fn']:,}, {_cm['tp']:,})")
print("\n✅ Section 6 complete.")


# =============================================================================
# SECTION 7: BOOTSTRAP CONFIDENCE INTERVALS -- HOLDOUT AUC AND PR-AUC
# =============================================================================
_section("SECTION 7: Bootstrap Confidence Intervals -- Holdout AUC and PR-AUC")

N_BOOTSTRAP = 2000
_rng = np.random.default_rng(RANDOM_SEED)
_n_holdout = len(y_holdout)
_boot_aucs = np.empty(N_BOOTSTRAP, dtype=np.float64)
_boot_pr_aucs = np.empty(N_BOOTSTRAP, dtype=np.float64)
for _b in range(N_BOOTSTRAP):
    _idx = _rng.integers(0, _n_holdout, size=_n_holdout)
    _yt, _yp = y_holdout[_idx], proba_holdout[_idx]
    if _yt.min() == _yt.max():
        _boot_aucs[_b] = np.nan  # degenerate resample (all-one-class); excluded below
        _boot_pr_aucs[_b] = np.nan
    else:
        _boot_aucs[_b] = roc_auc_score(_yt, _yp)
        _boot_pr_aucs[_b] = average_precision_score(_yt, _yp)
_valid_auc_boots = _boot_aucs[~np.isnan(_boot_aucs)]
_valid_pr_boots = _boot_pr_aucs[~np.isnan(_boot_pr_aucs)]
AUC_CI_LOWER, AUC_CI_UPPER = np.percentile(_valid_auc_boots, [2.5, 97.5])
PR_AUC_CI_LOWER, PR_AUC_CI_UPPER = np.percentile(_valid_pr_boots, [2.5, 97.5])
print(f"Bootstrap resamples: {N_BOOTSTRAP:,} (valid AUC: {len(_valid_auc_boots):,}, random_state={RANDOM_SEED})")
print(f"Holdout AUC 95% CI   : [{AUC_CI_LOWER:.4f}, {AUC_CI_UPPER:.4f}]  (point estimate {reproduced_auc:.4f})")
print(f"Holdout PR-AUC 95% CI: [{PR_AUC_CI_LOWER:.4f}, {PR_AUC_CI_UPPER:.4f}]  (point estimate {reproduced_pr_auc:.4f}, "
      f"no-skill baseline {holdout_positive_rate:.4f})")
_ci_excludes_random = AUC_CI_LOWER > 0.5
_pr_ci_excludes_noskill = PR_AUC_CI_LOWER > holdout_positive_rate
print(f"AUC 95% CI entirely above random (0.5) -- real, non-chance predictive power: {_ci_excludes_random}")
print(f"PR-AUC 95% CI entirely above the no-skill baseline (positive rate): {_pr_ci_excludes_noskill}")
print("\n✅ Section 7 complete.")


# =============================================================================
# SECTION 8: CALIBRATION CHECK -- PREDICTED PD DECILES VS. REAL OBSERVED DEFAULT RATE
# =============================================================================
_section("SECTION 8: Calibration Check")

_calib_df = pd.DataFrame({"proba": proba_holdout, "target": y_holdout})
_calib_df["decile"] = pd.qcut(_calib_df["proba"], q=10, labels=False, duplicates="drop")
calibration_table = (
    _calib_df.groupby("decile")
    .agg(n=("target", "size"), mean_predicted_pd=("proba", "mean"), observed_default_rate=("target", "mean"))
    .reset_index()
)
calibration_table["abs_gap"] = (calibration_table["mean_predicted_pd"] - calibration_table["observed_default_rate"]).abs()
print(calibration_table.round(4).to_string(index=False))
MEAN_CALIBRATION_GAP = float(calibration_table["abs_gap"].mean())
print(f"\nMean |predicted - observed| gap across deciles (real, measured): {MEAN_CALIBRATION_GAP:.4f}")
print("\n✅ Section 8 complete.")


# =============================================================================
# SECTION 9: SPLIT-HALF POPULATION STABILITY (PSI) ON THE PREDICTED SCORE
# =============================================================================
_section("SECTION 9: Split-Half Population Stability (PSI) on the Predicted Score")

# --- Same honest framing this platform already documents for Notebook 21/
#     monitoring_job.py's PSI (and reused verbatim by Notebook 36 for
#     Problem 5): a random split-half stability proxy on the single available
#     holdout population, not a genuine time-based drift measurement -- there
#     is no second real time period to compare against yet. Bin edges come
#     from the TRAIN score distribution's real deciles, applied to two random
#     halves of the holdout. ---
_train_edges = np.quantile(proba_train, np.linspace(0, 1, 11))
_train_edges[0], _train_edges[-1] = -np.inf, np.inf
_perm = _rng.permutation(_n_holdout)
_half = _n_holdout // 2
_half_a = proba_holdout[_perm[:_half]]
_half_b = proba_holdout[_perm[_half:]]
_share_a = np.histogram(_half_a, bins=_train_edges)[0] / len(_half_a)
_share_b = np.histogram(_half_b, bins=_train_edges)[0] / len(_half_b)
_share_a = np.clip(_share_a, 1e-4, None)
_share_b = np.clip(_share_b, 1e-4, None)
SCORE_PSI_SPLIT_HALF = float(((_share_a - _share_b) * np.log(_share_a / _share_b)).sum())
_psi_target = 0.10  # ASSUMPTION: standard industry PSI stability threshold (<0.10 = no significant shift)
print(f"Split-half PSI on predicted score: {SCORE_PSI_SPLIT_HALF:.4f}  (target < {_psi_target}, "
      f"{'PASS' if SCORE_PSI_SPLIT_HALF < _psi_target else 'FAIL'})")
print("\n✅ Section 9 complete.")


# =============================================================================
# SECTION 10: FULL METRICS-SUITE STATISTICAL VALIDATION SUMMARY TABLE
# =============================================================================
_section("SECTION 10: Full Metrics-Suite Statistical Validation Summary Table")

# --- Per the user's 2026-08-25 standing directive, the FULL classification
#     metrics suite is assembled into one table here (not just AUC) -- every
#     row below is a real, measured value from Section 6-9 above, none
#     fabricated or assumed. Threshold-dependent rows are reported at both
#     the standard 0.5 threshold and the F1-optimal threshold (same honesty
#     caveat as Notebook 39: F1-optimal was chosen on this same holdout). ---
_random_baseline_log_loss = 0.6931  # ln(2): log loss of a constant 0.5 prediction
statistical_validation_rows = [
    {"test": "Holdout AUC-ROC (reproduced)", "value": round(reproduced_auc, 4), "target": ">0.5", "pass": bool(reproduced_auc > 0.5)},
    {"test": "Holdout AUC-ROC 95% CI lower bound", "value": round(float(AUC_CI_LOWER), 4), "target": ">0.5", "pass": bool(_ci_excludes_random)},
    {"test": "Holdout PR-AUC (reproduced)", "value": round(reproduced_pr_auc, 4), "target": f">{holdout_positive_rate:.4f} (no-skill)", "pass": bool(reproduced_pr_auc > holdout_positive_rate)},
    {"test": "Holdout PR-AUC 95% CI lower bound", "value": round(float(PR_AUC_CI_LOWER), 4), "target": f">{holdout_positive_rate:.4f} (no-skill)", "pass": bool(_pr_ci_excludes_noskill)},
    {"test": "Holdout Log Loss (reproduced)", "value": round(reproduced_log_loss, 4), "target": f"<{_random_baseline_log_loss}", "pass": bool(reproduced_log_loss < _random_baseline_log_loss)},
    {"test": "Holdout AMEX competition metric (reproduced)", "value": round(reproduced_amex, 4), "target": ">0", "pass": bool(reproduced_amex > 0)},
    {"test": "Accuracy @ 0.50 threshold", "value": round(metrics_at_050["accuracy"], 4), "target": "reported", "pass": True},
    {"test": "Precision @ 0.50 threshold", "value": round(metrics_at_050["precision"], 4), "target": "reported", "pass": True},
    {"test": "Recall @ 0.50 threshold", "value": round(metrics_at_050["recall"], 4), "target": "reported", "pass": True},
    {"test": "F1 @ 0.50 threshold", "value": round(metrics_at_050["f1"], 4), "target": "reported", "pass": True},
    {"test": "Specificity @ 0.50 threshold", "value": round(metrics_at_050["specificity"], 4), "target": "reported", "pass": True},
    {"test": "MCC @ 0.50 threshold", "value": round(metrics_at_050["mcc"], 4), "target": ">0", "pass": bool(metrics_at_050["mcc"] > 0)},
    {"test": f"Accuracy @ F1-optimal threshold ({f1_optimal_threshold:.3f})", "value": round(metrics_at_f1_optimal["accuracy"], 4), "target": "reported", "pass": True},
    {"test": f"Precision @ F1-optimal threshold ({f1_optimal_threshold:.3f})", "value": round(metrics_at_f1_optimal["precision"], 4), "target": "reported", "pass": True},
    {"test": f"Recall @ F1-optimal threshold ({f1_optimal_threshold:.3f})", "value": round(metrics_at_f1_optimal["recall"], 4), "target": "reported", "pass": True},
    {"test": f"F1 @ F1-optimal threshold ({f1_optimal_threshold:.3f})", "value": round(metrics_at_f1_optimal["f1"], 4), "target": "reported", "pass": True},
    {"test": f"Specificity @ F1-optimal threshold ({f1_optimal_threshold:.3f})", "value": round(metrics_at_f1_optimal["specificity"], 4), "target": "reported", "pass": True},
    {"test": f"MCC @ F1-optimal threshold ({f1_optimal_threshold:.3f})", "value": round(metrics_at_f1_optimal["mcc"], 4), "target": ">0", "pass": bool(metrics_at_f1_optimal["mcc"] > 0)},
    {"test": "Mean calibration gap (deciles)", "value": round(MEAN_CALIBRATION_GAP, 4), "target": "<0.05", "pass": bool(MEAN_CALIBRATION_GAP < 0.05)},
    {"test": "Split-half score PSI", "value": round(SCORE_PSI_SPLIT_HALF, 4), "target": f"<{_psi_target}", "pass": bool(SCORE_PSI_SPLIT_HALF < _psi_target)},
    {"test": f"AUC retention vs. full history (W={WINNING_W})", "value": round(WINNING_W_RESULT["auc_retention_pct_of_full_history"], 1),
     "target": f">={KPI_TARGETS['min_auc_retention_vs_full_history']:.0%}", "pass": bool(MEETS_KPI)},
]
statistical_validation_df = pd.DataFrame(statistical_validation_rows)
statistical_validation_path = DBS_DEPLOYMENT_DIR / "dynamic_behavioral_statistical_validation.csv"
statistical_validation_df.to_csv(statistical_validation_path, index=False)
print(statistical_validation_df.to_string(index=False))
ALL_STAT_CHECKS_PASS = bool(statistical_validation_df["pass"].all())
print(f"\nAll statistical checks pass: {ALL_STAT_CHECKS_PASS}")
print(f"✅ Saved -> {statistical_validation_path}")
print("\n✅ Section 10 complete.")


# =============================================================================
# SECTION 11: HONEST LIMITATION -- DEPLOYMENT SCOPE & ASSUMPTIONS
# =============================================================================
_section("SECTION 11: Honest Limitation -- Deployment Scope & Assumptions")

DEPLOYMENT_LIMITATION = {
    "trailing_window_assumption": (
        f"This model was trained and validated on customers' MOST RECENT {WINNING_W} chronological "
        f"statements ONLY. It is a RECENCY score, not a replacement for Problem 1's full-history champion "
        f"-- both can coexist operationally: the full-history model for static book-wide PD, this model for "
        f"re-scoring using a customer's latest observed behavior. It assumes exactly (or at least) {WINNING_W} "
        f"real statements are available at scoring time; customers with fewer are out of scope for this model."
    ),
    "kpi_status": (
        f"MEETS the {KPI_TARGETS['min_auc_retention_vs_full_history']:.0%} AUC-retention KPI set in Notebook 38."
        if MEETS_KPI else
        f"DOES NOT MEET the {KPI_TARGETS['min_auc_retention_vs_full_history']:.0%} AUC-retention KPI set in "
        f"Notebook 38 -- this is the best-performing candidate window tested, packaged here for completeness "
        f"and so validation/deployment tooling exists, but it is NOT RECOMMENDED FOR PRODUCTION USE until a "
        f"future run either finds a viable window or the KPI target itself is revisited."
    ),
    "data_limitation": (
        "Same limitation Notebook 38 documented: this dataset has exactly one eventual-default label per "
        "customer, no month-by-month ground truth -- so this model still predicts the SAME real eventual-"
        "default outcome as Problem 1, only from a restricted (recent-only) feature snapshot, not a genuinely "
        "re-labeled 'default in the next N months' target."
    ),
    "feature_space_leakage_note": (
        "Reuses Problem 4's real, correlation-filtered feature list, re-aggregated per window from ONLY the "
        "trailing statements (not Problem 4's precomputed severity tier, which would leak full-history "
        "information into a short-window snapshot) -- see Notebook 38 Section 8."
    ),
    "single_architecture_scope": (
        f"Only the champion architecture ({CHAMPION_NAME}) was evaluated (see Notebook 39's Section 7 scope "
        "note) -- windowed GBM only; the LSTM alternative the master plan also names was explicitly deferred "
        "as a future extension in Notebook 38 Section 9, not built in this pass."
    ),
}
for _k, _v in DEPLOYMENT_LIMITATION.items():
    print(f"{_k}:\n  {_v}\n")
print("\n✅ Section 11 complete.")


# =============================================================================
# SECTION 12: PERSIST MODEL & PREPROCESSING ARTIFACTS
# =============================================================================
_section("SECTION 12: Persist Model & Preprocessing Artifacts")

MODEL_FILENAME = f"dynamic_behavioral_xgboost_w{WINNING_W}.joblib"
model_path = MODELS_SUBDIR / MODEL_FILENAME
joblib.dump(final_model, model_path)
print(f"✅ {model_path.name:<40} {model_path.stat().st_size / 1e6:>8,.2f} MB")

preprocessing_path = MODELS_SUBDIR / "preprocessing_artifacts.joblib"
joblib.dump({
    "feature_medians": feature_medians,
    "all_feature_cols": all_feature_cols,
    "base_feature_columns": BASE_FEATURE_COLUMNS,
    "w": WINNING_W,
    "model_filename": MODEL_FILENAME,
}, preprocessing_path)
print(f"✅ {preprocessing_path.name:<40} {preprocessing_path.stat().st_size / 1e6:>8,.2f} MB")
print("\n✅ Section 12 complete.")


# =============================================================================
# SECTION 13: GENERATE dynamic_behavioral_service.py -- REAL, RUNNABLE FASTAPI SERVICE
# =============================================================================
_section("SECTION 13: Generate dynamic_behavioral_service.py -- Real FastAPI Service")

# --- Same generation pattern Notebook 10 (Problem 1), Notebook 22 (Problem
#     2), and Notebook 36 (Problem 5) established: plain string-list build
#     (avoids f-string brace-escaping on this generated source's own literal
#     braces), same AMEX_PROJECT_ROOT-style env-var convention, model+
#     preprocessing paths baked in as tokens so the generated file is
#     genuinely standalone. This service's feature set is all-numeric D_*
#     (no categorical columns), simpler than Problem 5's service. ---
_models_subdir_str = str(MODELS_SUBDIR)

DYNAMIC_BEHAVIORAL_SERVICE_TEMPLATE = "\n".join([
    "# AMEX Enterprise Credit Risk Platform -- Dynamic/Behavioral Credit Scoring API.",
    "# Auto-generated by 40_dynamic_behavioral_scoring_validation_deployment.ipynb.",
    f"# Scores a customer using ONLY their most recent {WINNING_W} chronological statements -- see /model-info.",
    "# Run with:",
    "#     uvicorn dynamic_behavioral_service:app --host 0.0.0.0 --port 8003",
    "import os",
    "from pathlib import Path",
    "from typing import Optional",
    "",
    "import joblib",
    "import numpy as np",
    "from fastapi import FastAPI, HTTPException",
    "from pydantic import BaseModel, create_model",
    "",
    "MODELS_DIR = Path(os.environ.get(\"AMEX_DBS_MODELS_DIR\", r\"__MODELS_DIR_TOKEN__\"))",
    "",
    "preprocessing_artifacts = joblib.load(MODELS_DIR / \"preprocessing_artifacts.joblib\")",
    "feature_medians = preprocessing_artifacts[\"feature_medians\"]",
    "all_feature_cols = preprocessing_artifacts[\"all_feature_cols\"]",
    "TRAILING_WINDOW_W = preprocessing_artifacts[\"w\"]",
    "model = joblib.load(MODELS_DIR / preprocessing_artifacts[\"model_filename\"])",
    "",
    "_schema_fields = {_c: (Optional[float], None) for _c in all_feature_cols}",
    "CustomerFeatures = create_model(\"CustomerFeatures\", **_schema_fields)",
    "",
    "",
    "class DynamicBehavioralResponse(BaseModel):",
    "    customer_id: Optional[str] = None",
    "    predicted_pd: float",
    "    trailing_window_w: int",
    f"    meets_kpi_target: bool = {MEETS_KPI}",
    "",
    "",
    "app = FastAPI(",
    "    title=\"AMEX Enterprise Credit Risk Platform -- Dynamic/Behavioral Credit Scoring API\",",
    f"    description=\"Re-scores default risk using only a customer's most recent {WINNING_W} chronological \"",
    "                \"statements -- a RECENCY score, complementary to Problem 1's full-history champion, not a \"",
    "                \"replacement for it. See /model-info for the real validation metrics behind this model.\",",
    "    version=\"1.0.0\",",
    ")",
    "",
    "",
    "@app.get(\"/health\")",
    "def health():",
    "    return {\"status\": \"ok\", \"trailing_window_w\": TRAILING_WINDOW_W}",
    "",
    "",
    "@app.get(\"/model-info\")",
    "def model_info():",
    "    return {",
    "        \"trailing_window_w\": TRAILING_WINDOW_W,",
    f"        \"meets_kpi_target\": {MEETS_KPI},",
    f"        \"holdout_auc\": {WINNING_W_RESULT['holdout_auc']!r},",
    f"        \"holdout_pr_auc\": {WINNING_W_RESULT['holdout_pr_auc']!r},",
    f"        \"holdout_amex_metric\": {WINNING_W_RESULT['holdout_amex_metric']!r},",
    f"        \"auc_retention_pct_of_full_history\": {WINNING_W_RESULT['auc_retention_pct_of_full_history']!r},",
    f"        \"recommended_for_production\": {MEETS_KPI},",
    "    }",
    "",
    "",
    "@app.post(\"/score\", response_model=DynamicBehavioralResponse)",
    "def score(features: CustomerFeatures, customer_id: Optional[str] = None):",
    "    row = features.dict() if hasattr(features, \"dict\") else features.model_dump()",
    "    x = np.zeros((1, len(all_feature_cols)), dtype=np.float32)",
    "    for i, col in enumerate(all_feature_cols):",
    "        val = row.get(col)",
    "        if val is None or (isinstance(val, float) and np.isnan(val)):",
    "            val = feature_medians[col]",
    "        x[0, i] = val",
    "    try:",
    "        pd_score = float(model.predict_proba(x)[:, 1][0])",
    "    except Exception as exc:",
    "        raise HTTPException(status_code=500, detail=\"Scoring failed: \" + str(exc))",
    "    return DynamicBehavioralResponse(customer_id=customer_id, predicted_pd=pd_score, trailing_window_w=TRAILING_WINDOW_W)",
    "",
])
DYNAMIC_BEHAVIORAL_SERVICE_SOURCE = DYNAMIC_BEHAVIORAL_SERVICE_TEMPLATE.replace("__MODELS_DIR_TOKEN__", _models_subdir_str)

service_py_path = API_SUBDIR / "dynamic_behavioral_service.py"
with open(service_py_path, "w", encoding="utf-8") as f:
    f.write(DYNAMIC_BEHAVIORAL_SERVICE_SOURCE)
compile(DYNAMIC_BEHAVIORAL_SERVICE_SOURCE, str(service_py_path), "exec")
print(f"Generated {len(DYNAMIC_BEHAVIORAL_SERVICE_SOURCE.splitlines())} lines, syntax-checked OK.")
print(f"✅ Saved -> {service_py_path}")
print("\n✅ Section 13 complete.")


# =============================================================================
# SECTION 14: GENERATE .env.example & requirements-api.txt
# =============================================================================
_section("SECTION 14: Generate .env.example & requirements-api.txt")

ENV_EXAMPLE = f"""# Copy to .env and edit if this machine's models folder differs from the default.
AMEX_DBS_MODELS_DIR={MODELS_SUBDIR}
"""
env_example_path = API_SUBDIR / ".env.example"
with open(env_example_path, "w", encoding="utf-8") as f:
    f.write(ENV_EXAMPLE)

_api_packages = ["fastapi", "uvicorn", "pydantic", "joblib", "numpy", "scikit-learn", "xgboost"]
_api_pkg_versions = {}
for _pkg in _api_packages:
    try:
        _api_pkg_versions[_pkg] = importlib_metadata.version(_pkg)
    except importlib_metadata.PackageNotFoundError:
        _api_pkg_versions[_pkg] = None

requirements_api_path = API_SUBDIR / "requirements-api.txt"
with open(requirements_api_path, "w", encoding="utf-8") as f:
    f.write(f"# Minimal runtime dependencies for dynamic_behavioral_service.py -- auto-generated "
             f"{datetime.now().strftime('%Y-%m-%d %H:%M')}\n")
    for _pkg, _ver in _api_pkg_versions.items():
        f.write(f"{_pkg}=={_ver}\n" if _ver else f"# {_pkg}  -- not installed here\n")

print(f"✅ Saved -> {env_example_path}")
print(f"✅ Saved -> {requirements_api_path}")
print("\n✅ Section 14 complete.")


# =============================================================================
# SECTION 15: LIVE SELF-TEST -- IMPORT THE GENERATED SERVICE & DRIVE IT
# =============================================================================
_section("SECTION 15: Live Self-Test -- Import the Generated Service & Drive It")

# --- Imports the EXACT file just written to disk -- proves the delivered
#     artifact works, not just an in-notebook copy of the same logic. ---
os.environ["AMEX_DBS_MODELS_DIR"] = str(MODELS_SUBDIR)
_spec = importlib.util.spec_from_file_location("amex_dynamic_behavioral_service", str(service_py_path))
_service_module = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_service_module)
client = TestClient(_service_module.app)

_health_resp = client.get("/health")
assert _health_resp.status_code == 200, f"/health returned {_health_resp.status_code}"
print(f"GET /health     -> {_health_resp.status_code}  {_health_resp.json()}")

_info_resp = client.get("/model-info")
assert _info_resp.status_code == 200, f"/model-info returned {_info_resp.status_code}"
print(f"GET /model-info -> {_info_resp.status_code}  {_info_resp.json()}")

_sample_idx = 0
SAMPLE_CUSTOMER_ID = holdout_customer_ids[_sample_idx]
# --- All features here are numeric D_* (no categorical encoding, unlike
#     Problem 5's service) -- the payload is a direct float dump of the
#     already-imputed X_holdout row. ---
SAMPLE_PAYLOAD = {}
for _i, _col in enumerate(all_feature_cols):
    _val = X_holdout[_sample_idx, _i]
    SAMPLE_PAYLOAD[_col] = None if (isinstance(_val, float) and np.isnan(_val)) else float(_val)
EXPECTED_PD_DIRECT = float(proba_holdout[_sample_idx])

_score_resp = client.post("/score", params={"customer_id": SAMPLE_CUSTOMER_ID}, json=SAMPLE_PAYLOAD)
assert _score_resp.status_code == 200, f"/score returned {_score_resp.status_code}: {_score_resp.text}"
_api_result = _score_resp.json()
_api_pd = _api_result["predicted_pd"]
print(f"POST /score     -> {_score_resp.status_code}  {_api_result}")

_pd_diff = abs(_api_pd - EXPECTED_PD_DIRECT)
print(f"\nEnd-to-end check: API PD ({_api_pd:.6f}) vs. directly-computed PD ({EXPECTED_PD_DIRECT:.6f}) -- diff {_pd_diff:.8f}")

API_SELF_TEST_PASSED = _pd_diff < 1e-3
if API_SELF_TEST_PASSED:
    print("\n✅ MATCH -- the live API's preprocessing and scoring are verified consistent with direct computation.")
else:
    print("\n❌ MISMATCH -- do not deploy dynamic_behavioral_service.py until this is resolved.")

if not API_SELF_TEST_PASSED:
    raise RuntimeError("Notebook 40's API self-test FAILED -- see ❌ line above. Not safe to proceed.")
print("\n✅ Section 15 complete.")


# =============================================================================
# SECTION 16: API LATENCY BENCHMARK
# =============================================================================
_section("SECTION 16: API Latency Benchmark")

N_API_LATENCY_SAMPLES = 150
_api_latencies_ms = []
for _ in range(N_API_LATENCY_SAMPLES):
    _t0 = time.perf_counter()
    _ = client.post("/score", json=SAMPLE_PAYLOAD)
    _api_latencies_ms.append((time.perf_counter() - _t0) * 1000.0)
_api_latencies_ms = np.array(_api_latencies_ms)
api_latency_summary = {
    "n_samples": N_API_LATENCY_SAMPLES,
    "p50_ms": round(float(np.percentile(_api_latencies_ms, 50)), 3),
    "p95_ms": round(float(np.percentile(_api_latencies_ms, 95)), 3),
    "p99_ms": round(float(np.percentile(_api_latencies_ms, 99)), 3),
    "max_ms": round(float(_api_latencies_ms.max()), 3),
}
print(f"/score latency over {N_API_LATENCY_SAMPLES} real TestClient calls: {api_latency_summary}")
print("\n✅ Section 16 complete.")


# =============================================================================
# SECTION 17: DEPLOYMENT READINESS CHECKLIST
# =============================================================================
_section("SECTION 17: Deployment Readiness Checklist")

deployment_readiness_rows = [
    {"dimension": "Model reproducibility", "status": "PASS" if _reproduction_matches else "FAIL"},
    {"dimension": "Full metrics-suite statistical validation (all checks)", "status": "PASS" if ALL_STAT_CHECKS_PASS else "FAIL"},
    {"dimension": "AUC-retention KPI (Notebook 38 target)", "status": "MET" if MEETS_KPI else "NOT MET"},
    {"dimension": "Model + preprocessing artifacts persisted", "status": "PASS" if model_path.exists() and preprocessing_path.exists() else "FAIL"},
    {"dimension": "API self-test (live, generated service)", "status": "PASS" if API_SELF_TEST_PASSED else "FAIL"},
    {"dimension": "API p99 latency < 500ms", "status": "PASS" if api_latency_summary["p99_ms"] < 500 else "FAIL"},
    {"dimension": "Overall recommendation", "status": "RECOMMENDED FOR PRODUCTION" if MEETS_KPI and ALL_STAT_CHECKS_PASS else "NOT RECOMMENDED FOR PRODUCTION"},
]
deployment_readiness_df = pd.DataFrame(deployment_readiness_rows)
deployment_readiness_path = DBS_DEPLOYMENT_DIR / "deployment_readiness_checklist.csv"
deployment_readiness_df.to_csv(deployment_readiness_path, index=False)
print(deployment_readiness_df.to_string(index=False))
print(f"✅ Saved -> {deployment_readiness_path}")
print("\n✅ Section 17 complete.")


# =============================================================================
# SECTION 18: CHARTS
# =============================================================================
_section("SECTION 18: Charts")

CHARTS_DIR = DBS_DEPLOYMENT_DIR / "charts"
CHARTS_DIR.mkdir(parents=True, exist_ok=True)


def _style_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", alpha=0.3)


fig, ax = plt.subplots(figsize=(7, 4.5))
_ws_sorted = sorted(RESULTS_BY_W.keys())
_aucs = [RESULTS_BY_W[w]["holdout_auc"] for w in _ws_sorted]
ax.plot(_ws_sorted, _aucs, marker="o", linewidth=2, color="#2563eb", label="Trailing-window holdout AUC")
ax.axhline(FULL_HISTORY_AUC, color="#64748b", linestyle="--", label=f"Full-history AUC ({FULL_HISTORY_AUC:.4f})")
ax.axhline(FULL_HISTORY_AUC * KPI_TARGETS["min_auc_retention_vs_full_history"], color="#dc2626", linestyle=":",
           label=f"KPI floor ({KPI_TARGETS['min_auc_retention_vs_full_history']:.0%} retention)")
ax.scatter([WINNING_W], [WINNING_W_RESULT["holdout_auc"]], color="#16a34a", s=100, zorder=5, label=f"Selected: W={WINNING_W}")
ax.set_xlabel("Trailing window W (most recent statements)")
ax.set_ylabel("Holdout AUC")
ax.set_title("AUC-Retention Curve -- Trailing Window vs. Full History")
ax.legend(fontsize=8)
_style_axes(ax)
chart1_path = CHARTS_DIR / "auc_retention_curve.png"
fig.tight_layout()
fig.savefig(chart1_path, dpi=150)
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(calibration_table["mean_predicted_pd"], calibration_table["observed_default_rate"],
        marker="o", linewidth=2, color="#2563eb", label=f"W={WINNING_W} model (real deciles)")
_diag = [0, max(calibration_table["mean_predicted_pd"].max(), calibration_table["observed_default_rate"].max())]
ax.plot(_diag, _diag, color="#94a3b8", linestyle="--", label="Perfect calibration")
ax.set_xlabel("Mean predicted PD (decile)")
ax.set_ylabel("Observed default rate (decile)")
ax.set_title(f"Calibration -- W={WINNING_W} Trailing-Window Model")
ax.legend(fontsize=8)
_style_axes(ax)
chart2_path = CHARTS_DIR / "calibration_curve.png"
fig.tight_layout()
fig.savefig(chart2_path, dpi=150)
plt.show()
plt.close(fig)

print(f"✅ Saved -> {chart1_path}")
print(f"✅ Saved -> {chart2_path}")
print(f"(Reusing Notebook 39's ROC/PR curve charts in the report below -- not regenerated: "
      f"{NB39_ROC_CHART_PATH.name}, {NB39_PR_CHART_PATH.name})")
print("\n✅ Section 18 complete.")


# =============================================================================
# SECTION 19: WORD REPORT
# =============================================================================
_section("SECTION 19: Word Report -- Dynamic_Behavioral_Validation_Deployment_Report.docx")

doc = Document()
doc.add_heading("AMEX Enterprise Credit Risk Platform", level=0)
doc.add_paragraph("Phase 3, Problem 6: Dynamic / Behavioral Credit Scoring -- Validation & Deployment Report")
doc.add_paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

doc.add_heading("1. Scope & Selected Window", level=1)
doc.add_paragraph(
    f"Statistical validation and deployment packaging for the trailing-window model at W={WINNING_W} "
    f"most-recent statements, selected from Notebook 39's real AUC-retention curve across candidates "
    f"{sorted(RESULTS_BY_W.keys())}. "
    + ("This window meets Notebook 38's AUC-retention KPI target." if MEETS_KPI else
       "IMPORTANT: none of the tested candidates met Notebook 38's AUC-retention KPI target on this real "
       "run -- this is the best-performing candidate, packaged for completeness, and is NOT RECOMMENDED "
       "FOR PRODUCTION until a future run finds a viable window.")
)

doc.add_heading("2. Full Classification Metrics-Suite Validation Summary", level=1)
doc.add_paragraph(
    "Per the platform's standing metrics-suite directive (effective Problem 6 onward): every metric below "
    "is a real, measured value reproduced independently in this notebook, cross-checked against Notebook "
    "39's originally-reported numbers (see Section 6's reproduction check)."
)
_t = doc.add_table(rows=1, cols=len(statistical_validation_df.columns))
_t.style = "Light Grid Accent 1"
for _i, _col in enumerate(statistical_validation_df.columns):
    _t.rows[0].cells[_i].text = _col.replace("_", " ").title()
for _, _row in statistical_validation_df.iterrows():
    _cells = _t.add_row().cells
    for _i, _col in enumerate(statistical_validation_df.columns):
        _cells[_i].text = str(_row[_col])

doc.add_heading("3. Honest Limitation -- Deployment Scope & Assumptions", level=1)
for _k, _v in DEPLOYMENT_LIMITATION.items():
    doc.add_paragraph(f"{_k.replace('_', ' ').title()}: {_v}")

doc.add_heading("4. Deployment Readiness Checklist", level=1)
_t3 = doc.add_table(rows=1, cols=len(deployment_readiness_df.columns))
_t3.style = "Light Grid Accent 1"
for _i, _col in enumerate(deployment_readiness_df.columns):
    _t3.rows[0].cells[_i].text = _col.replace("_", " ").title()
for _, _row in deployment_readiness_df.iterrows():
    _cells = _t3.add_row().cells
    for _i, _col in enumerate(deployment_readiness_df.columns):
        _cells[_i].text = str(_row[_col])

doc.add_heading("5. API Performance", level=1)
doc.add_paragraph(f"Latency over {api_latency_summary['n_samples']} real TestClient calls to /score: "
                   f"p50={api_latency_summary['p50_ms']}ms, p95={api_latency_summary['p95_ms']}ms, "
                   f"p99={api_latency_summary['p99_ms']}ms, max={api_latency_summary['max_ms']}ms.")

doc.add_heading("6. Charts", level=1)
_chart_entries = [
    (chart1_path, "AUC-retention curve across candidate trailing windows"),
    (chart2_path, f"Calibration -- W={WINNING_W} model, real holdout deciles"),
]
if NB39_ROC_CHART_PATH.exists():
    _chart_entries.append((NB39_ROC_CHART_PATH, "ROC curves by trailing window (from Notebook 39)"))
if NB39_PR_CHART_PATH.exists():
    _chart_entries.append((NB39_PR_CHART_PATH, "Precision-Recall curves by trailing window (from Notebook 39)"))
for _cp, _cap in _chart_entries:
    doc.add_picture(str(_cp), width=Inches(6.0))
    _p = doc.add_paragraph(_cap)
    _p.alignment = WD_ALIGN_PARAGRAPH.CENTER

report_path = DBS_DEPLOYMENT_DIR / "Dynamic_Behavioral_Validation_Deployment_Report.docx"
doc.save(report_path)
print(f"✅ Saved -> {report_path}")
print("\n✅ Section 19 complete.")


# =============================================================================
# SECTION 20: VERIFICATION -- INTEGRITY CHECKS
# =============================================================================
_section("SECTION 20: Verification -- Integrity Checks")


def _check(label, condition, detail=""):
    status = "PASS" if condition else "FAIL"
    print(f"  [{status}] {label}" + (f" -- {detail}" if detail and not condition else ""))
    return condition


_all_checks_passed = True
_all_checks_passed &= _check("Model reproduction matches Notebook 39's reported holdout AUC and PR-AUC", _reproduction_matches)
_all_checks_passed &= _check("Bootstrap AUC CI is well-formed (lower <= point estimate <= upper)",
                              AUC_CI_LOWER <= reproduced_auc <= AUC_CI_UPPER)
_all_checks_passed &= _check("Bootstrap PR-AUC CI is well-formed (lower <= point estimate <= upper)",
                              PR_AUC_CI_LOWER <= reproduced_pr_auc <= PR_AUC_CI_UPPER)
_all_checks_passed &= _check("Calibration table has 10 (or fewer, if ties) deciles", 1 <= len(calibration_table) <= 10)
_all_checks_passed &= _check(
    "F1 at the F1-optimal threshold is >= F1 at the 0.5 threshold "
    "(the optimal threshold was chosen BY maximizing F1 on this same holdout set)",
    metrics_at_f1_optimal["f1"] >= metrics_at_050["f1"] - 1e-9,
)
_all_checks_passed &= _check(
    "Both confusion matrices (0.5 and F1-optimal) sum to the real holdout customer count",
    sum(metrics_at_050["confusion_matrix"].values()) == len(y_holdout)
    and sum(metrics_at_f1_optimal["confusion_matrix"].values()) == len(y_holdout),
)
_all_checks_passed &= _check("API self-test passed", API_SELF_TEST_PASSED)
_expected_files = [statistical_validation_path, deployment_readiness_path, model_path, preprocessing_path,
                    service_py_path, env_example_path, requirements_api_path, chart1_path, chart2_path, report_path]
for _fp in _expected_files:
    _all_checks_passed &= _check(f"{_fp.name} exists and is non-empty", _fp.exists() and _fp.stat().st_size > 0)

if not _all_checks_passed:
    raise AssertionError("One or more verification checks failed -- see FAIL lines above.")
print("\n✅ Section 20 complete -- all checks passed.")


# =============================================================================
# SECTION 21: WRITE NOTEBOOK 40 SUMMARY ARTIFACT & COMPLETION
# =============================================================================
_section("SECTION 21: Write Notebook 40 Summary Artifact")

NB40_SUMMARY = {
    "notebook": "40_dynamic_behavioral_scoring_validation_deployment.ipynb",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "winning_w": WINNING_W,
    "meets_kpi_target": MEETS_KPI,
    "recommended_for_production": bool(MEETS_KPI and ALL_STAT_CHECKS_PASS),
    "reproduced_holdout_auc": reproduced_auc,
    "reproduced_holdout_pr_auc": reproduced_pr_auc,
    "reproduced_holdout_log_loss": reproduced_log_loss,
    "reproduced_holdout_amex_metric": reproduced_amex,
    "metrics_at_threshold_0_50": metrics_at_050,
    "metrics_at_f1_optimal_threshold": metrics_at_f1_optimal,
    "bootstrap_auc_ci": [float(AUC_CI_LOWER), float(AUC_CI_UPPER)],
    "bootstrap_pr_auc_ci": [float(PR_AUC_CI_LOWER), float(PR_AUC_CI_UPPER)],
    "mean_calibration_gap": MEAN_CALIBRATION_GAP,
    "split_half_score_psi": SCORE_PSI_SPLIT_HALF,
    "model_path": str(model_path),
    "preprocessing_path": str(preprocessing_path),
    "service_py_path": str(service_py_path),
    "report_path": str(report_path),
    "statistical_validation_path": str(statistical_validation_path),
    "random_seed": RANDOM_SEED,
}
NB40_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_40_summary.json"
with open(NB40_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(NB40_SUMMARY, f, indent=2)
print(f"Wrote: {NB40_SUMMARY_PATH}")

_section("NOTEBOOK 40 COMPLETE")
print(f"Winning window             : W={WINNING_W}")
print(f"Meets KPI target            : {MEETS_KPI}")
print(f"Recommended for production  : {NB40_SUMMARY['recommended_for_production']}")
print(f"Reproduced holdout AUC      : {reproduced_auc:.4f}  (95% CI [{AUC_CI_LOWER:.4f}, {AUC_CI_UPPER:.4f}])")
print(f"Reproduced holdout PR-AUC   : {reproduced_pr_auc:.4f}  (95% CI [{PR_AUC_CI_LOWER:.4f}, {PR_AUC_CI_UPPER:.4f}])")
print(f"API self-test               : {'PASSED' if API_SELF_TEST_PASSED else 'FAILED'}")
print(f"Word report                 : {report_path}")
print(
    "\nNext: 41_dynamic_behavioral_scoring_financial_impact_reporting_packaging.ipynb -- financial-impact "
    "reporting and final packaging for Problem 6, closing out Problem 6 of Phase 3."
)
